In [ ]:
import requests, csv, os

## 1. Data Collection

This section pulls live job posting data from the **Adzuna API** to analyze which technical skills are currently in demand.

- **Countries:** US, India, and the UK — chosen to capture demand signals across different regional markets
- **Search keywords:** 10 role titles (e.g. Python Developer, Data Scientist, DevOps Engineer, Full Stack Developer) covering a spread of common technical roles
- **Volume:** up to 50 results per page, 2 pages per keyword per country
- **Deduplication:** postings are deduplicated by their unique `id` so re-running the collection doesn't create duplicate rows

The output is a raw CSV of job postings with title, company, location, salary range, category, and full description text — this is the input for the cleaning and extraction step below.

In [ ]:
APP_ID = "50d2c099"
APP_KEY = "ENTER_APP_KEY_HERE"
COUNTRIES = ["us", "in", "gb"]
KEYWORDS = ["python developer", "react developer", "aws engineer", "javascript developer",
            "data scientist", "devops engineer", "java developer", "sql developer",
            "machine learning engineer", "full stack developer"]

OUT_FILE = "/content/adzuna_jobs.csv"
existing_ids = set()
if os.path.exists(OUT_FILE):
    with open(OUT_FILE, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            existing_ids.add(row["id"])

rows = []
for country in COUNTRIES:
    for kw in KEYWORDS:
        for page in range(1, 3):
            url = f"https://api.adzuna.com/v1/api/jobs/{country}/search/{page}"
            params = {"app_id": APP_ID, "app_key": APP_KEY, "what": kw, "results_per_page": 50}
            r = requests.get(url, params=params)
            if r.status_code != 200:
                print(f"Skipped {country}/{kw} page {page}: {r.status_code}")
                continue
            for job in r.json().get("results", []):
                jid = str(job.get("id"))
                if jid in existing_ids:
                    continue
                existing_ids.add(jid)
                rows.append({
                    "id": jid,
                    "country": country,
                    "title": job.get("title", ""),
                    "company": job.get("company", {}).get("display_name", ""),
                    "location": job.get("location", {}).get("display_name", ""),
                    "salary_min": job.get("salary_min", ""),
                    "salary_max": job.get("salary_max", ""),
                    "category": job.get("category", {}).get("label", ""),
                    "created": job.get("created", ""),
                    "description": job.get("description", "").replace("\n", " "),
                    "url": job.get("redirect_url", ""),
                    "keyword_matched": kw
                })

file_exists = os.path.exists(OUT_FILE)
with open(OUT_FILE, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id","country","title","company","location","salary_min",
                                            "salary_max","category","created","description",
                                            "url","keyword_matched"])
    if not file_exists:
        writer.writeheader()
    writer.writerows(rows)

print(f"Added {len(rows)} new rows. Total unique so far: {len(existing_ids)}")

Added 2959 new rows. Total unique so far: 2959


In [ ]:
from google.colab import files
files.download("/content/adzuna_jobs.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 2. Data Cleaning & Skill Extraction

Rather than relying on free-text NLP, this project uses a **whitelist-based regex approach**: ~40 common tech skills (languages, frameworks, cloud platforms, databases, tools) are defined as regex patterns and matched against each posting's title + description text.

- **Why regex over NLP:** at this scale, a curated whitelist is more precise and interpretable — every match is traceable to an explicit pattern, with no ambiguity about what triggered it
- **Word-boundary matching (`\b`)** is used throughout to avoid partial-word false positives (e.g. "java" not matching inside "javascript")
- **Known limitation:** a whitelist can only detect skills it explicitly knows about — anything outside the ~40 tracked skills is invisible to this analysis. Patterns involving symbols (like C# and C++) also needed extra care, since standard `\b` boundaries don't behave as expected around non-word characters

The script outputs three files: the original data enriched with a per-row skill list, a skill frequency table, and a skill co-occurrence (pairwise) table.

In [ ]:

import pandas as pd
import re
from itertools import combinations
from collections import Counter

INPUT_FILE = "/content/adzuna_jobs.csv"
OUTPUT_DIR = "/content"

df = pd.read_csv(INPUT_FILE)

SKILL_PATTERNS = {
    "python": r"\bpython\b",
    "javascript": r"\bjavascript\b|\bjs\b",
    "typescript": r"\btypescript\b",
    "react": r"\breact(?:\.js)?\b",
    "angular": r"\bangular(?:js)?\b",
    "vue": r"\bvue(?:\.js)?\b",
    "node": r"\bnode(?:\.js)?\b",
    "django": r"\bdjango\b",
    "flask": r"\bflask\b",
    "java": r"\bjava\b(?!script)",
    "c#": r"\bc#(?!\w)",
    "c++": r"\bc\+\+(?!\w)",
    "go": r"\bgolang\b|\bgo\s?lang\b",
    "aws": r"\baws\b|\bamazon web services\b",
    "azure": r"\bazure\b",
    "gcp": r"\bgcp\b|\bgoogle cloud\b",
    "docker": r"\bdocker\b",
    "kubernetes": r"\bkubernetes\b|\bk8s\b",
    "terraform": r"\bterraform\b",
    "sql": r"\bsql\b",
    "mysql": r"\bmysql\b",
    "postgresql": r"\bpostgres(?:ql)?\b",
    "mongodb": r"\bmongodb\b|\bmongo\b",
    "nosql": r"\bnosql\b",
    "redis": r"\bredis\b",
    "spark": r"\bspark\b",
    "hadoop": r"\bhadoop\b",
    "kafka": r"\bkafka\b",
    "machine learning": r"\bmachine learning\b|\bml\b",
    "deep learning": r"\bdeep learning\b",
    "tensorflow": r"\btensorflow\b",
    "pytorch": r"\bpytorch\b",
    "pandas": r"\bpandas\b",
    "git": r"\bgit\b",
    "ci/cd": r"\bci/cd\b|\bci\-cd\b|\bcontinuous integration\b",
    "jenkins": r"\bjenkins\b",
    "rest api": r"\brest(?:ful)?\s?api\b",
    "graphql": r"\bgraphql\b",
    "html": r"\bhtml5?\b",
    "css": r"\bcss3?\b",
    "linux": r"\blinux\b",
    "power bi": r"\bpower\s?bi\b",
    "tableau": r"\btableau\b",
    "excel": r"\bexcel\b",
}


def extract_skills(row):
    text = f"{row['title']} {row['description']}".lower()
    return [skill for skill, pat in SKILL_PATTERNS.items() if re.search(pat, text)]


df["skills"] = df.apply(extract_skills, axis=1)
df["skill_count"] = df["skills"].apply(len)
df["skills_str"] = df["skills"].apply(lambda s: ";".join(s))

out_cols = [c for c in df.columns if c not in ("skills",)]
df[out_cols].to_csv(f"{OUTPUT_DIR}/adzuna_jobs_with_skills.csv", index=False)

# --- Skill frequency ---
all_skills = Counter(s for skills in df["skills"] for s in skills)
freq_df = pd.DataFrame(all_skills.most_common(), columns=["skill", "count"])
freq_df.to_csv(f"{OUTPUT_DIR}/skill_frequency.csv", index=False)

# --- Skill co-occurrence pairs ---
pair_counter = Counter()
for skills in df["skills"]:
    for a, b in combinations(sorted(set(skills)), 2):
        pair_counter[(a, b)] += 1
pairs_df = pd.DataFrame(
    [(a, b, c) for (a, b), c in pair_counter.most_common()],
    columns=["skill_a", "skill_b", "count"],
)
pairs_df.to_csv(f"{OUTPUT_DIR}/skill_pairs.csv", index=False)

print("Rows:", len(df))
print("Rows with >=1 skill detected:", (df["skill_count"] > 0).sum())
print("\nTop 15 skills:")
print(freq_df.head(15).to_string(index=False))
print("\nTop 15 skill pairs:")
print(pairs_df.head(15).to_string(index=False))

Rows: 2959
Rows with >=1 skill detected: 2335

Top 15 skills:
           skill  count
          python    451
             aws    451
            java    425
           react    419
      javascript    418
machine learning    413
             sql    403
           ci/cd    155
      typescript    126
           azure    104
            html     99
            node     93
             css     91
      kubernetes     84
         angular     82

Top 15 skill pairs:
   skill_a    skill_b  count
javascript      react    227
javascript typescript     94
javascript       node     90
       aws     python     88
     react typescript     88
      html javascript     87
       css       html     83
       css javascript     82
    python        sql     78
       aws      ci/cd     68
      node      react     58
      java      react     57
      html      react     56
       css      react     55
      java javascript     52


## 3. Analysis & Visualization

With skills extracted, this section turns the frequency and co-occurrence tables into visuals that answer three questions:

- **`top_skills.png`** — which individual skills appear most often across postings (raw demand ranking)
- **`skill_heatmap.png`** — which skills tend to be requested *together* in the same posting, restricted to the top 15 skills for readability (not because other skills are unimportant)
- **`salary_by_skill.png`** — whether specific skills correlate with higher advertised salaries

Together these move from "what's popular" to "what's popular *and* paired together" to "what's popular and pays well" — three different lenses on the same underlying data.

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

JOBS_FILE = "adzuna_jobs_with_skills.csv"
FREQ_FILE = "skill_frequency.csv"
PAIRS_FILE = "skill_pairs.csv"
TOP_N = 15

df = pd.read_csv(JOBS_FILE)
freq_df = pd.read_csv(FREQ_FILE)
pairs_df = pd.read_csv(PAIRS_FILE)

# ------ Top skills bar chart--------

top_skills = freq_df.head(TOP_N)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_skills, x="count", y="skill", palette="viridis")
plt.title(f"Top {TOP_N} In-Demand Tech Skills")
plt.xlabel("Number of job postings mentioning skill")
plt.ylabel("Skill")
plt.tight_layout()
plt.savefig("top_skills.png", dpi=150)
plt.close()


# -------- Co-occurrence heatmap (restricted to top skills for readability)-------

top_skill_names = top_skills["skill"].tolist()
matrix = pd.DataFrame(0, index=top_skill_names, columns=top_skill_names)

for _, row in pairs_df.iterrows():
    a, b, c = row["skill_a"], row["skill_b"], row["count"]
    if a in top_skill_names and b in top_skill_names:
        matrix.loc[a, b] = c
        matrix.loc[b, a] = c

plt.figure(figsize=(10, 8))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", square=True, cbar_kws={"label": "Co-occurrence count"})
plt.title(f"Skill Co-occurrence Heatmap (Top {TOP_N} Skills)")
plt.tight_layout()
plt.savefig("skill_heatmap.png", dpi=150)
plt.close()


# ------ Salary by skill (optional - only if salary_max column has usable data)-----

has_salary = "salary_max" in df.columns and df["salary_max"].notna().sum() > 0

if has_salary:
    df["skills_list"] = df["skills_str"].fillna("").apply(lambda s: s.split(";") if s else [])
    exploded = df.explode("skills_list")
    exploded = exploded[exploded["skills_list"].isin(top_skill_names[:8])]  # keep it readable
    exploded = exploded.dropna(subset=["salary_max"])

    if not exploded.empty:
        plt.figure(figsize=(10, 6))
        order = exploded.groupby("skills_list")["salary_max"].median().sort_values(ascending=False).index
        sns.boxplot(data=exploded, x="skills_list", y="salary_max", order=order, palette="Set2")
        plt.title("Salary Distribution by Skill (Top 8 Skills)")
        plt.xlabel("Skill")
        plt.ylabel("Salary (max)")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.savefig("salary_by_skill.png", dpi=150)
        plt.close()
    else:
        has_salary = False


# ------ Auto-generated key findings summary------

top_skill_name = freq_df.iloc[0]["skill"]
top_skill_count = int(freq_df.iloc[0]["count"])
total_postings = len(df)
top_pair = pairs_df.iloc[0]
skills_detected_pct = (df["skill_count"] > 0).mean() * 100 if "skill_count" in df.columns else None

lines = []
lines.append("KEY FINDINGS")
lines.append("=" * 40)
lines.append(f"- Analyzed {total_postings} job postings.")
if skills_detected_pct is not None:
    lines.append(f"- {skills_detected_pct:.1f}% of postings mentioned at least one whitelisted skill.")
lines.append(f"- Most in-demand skill: '{top_skill_name}' ({top_skill_count} postings).")
lines.append(f"- Top 3 skills overall: {', '.join(top_skills['skill'].head(3).tolist())}.")
lines.append(f"- Most common skill pairing: '{top_pair['skill_a']}' + '{top_pair['skill_b']}' "
             f"({int(top_pair['count'])} postings mention both).")
if has_salary:
    lines.append("- Salary varies noticeably by skill — see salary_by_skill.png for the breakdown.")
lines.append("")
lines.append("Charts saved: top_skills.png, skill_heatmap.png"
             + (", salary_by_skill.png" if has_salary else ""))

summary = "\n".join(lines)
with open("key_findings.txt", "w") as f:
    f.write(summary)

print(summary)

/tmp/ipykernel_1038/3208452723.py:35: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=top_skills, x="count", y="skill", palette="viridis")
/tmp/ipykernel_1038/3208452723.py:72: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=exploded, x="skills_list", y="salary_max", order=order, palette="Set2")


KEY FINDINGS
- Analyzed 2959 job postings.
- 78.9% of postings mentioned at least one whitelisted skill.
- Most in-demand skill: 'python' (451 postings).
- Top 3 skills overall: python, aws, java.
- Most common skill pairing: 'javascript' + 'react' (227 postings mention both).
- Salary varies noticeably by skill — see salary_by_skill.png for the breakdown.

Charts saved: top_skills.png, skill_heatmap.png, salary_by_skill.png


KEY FINDINGS
========================================
- Analyzed 2959 job postings.
- 78.9% of postings mentioned at least one whitelisted skill.
- Most in-demand skill: 'python' (451 postings).
- Top 3 skills overall: python, aws, java.
- Most common skill pairing: 'javascript' + 'react' (227 postings mention both).
- Salary varies noticeably by skill.

In [6]:
from google.colab import files
for f in ["adzuna_jobs_with_skills.csv", "skill_frequency.csv", "skill_pairs.csv", "top_skills.png", "skill_heatmap.png", "salary_by_skill.png", "key_findings.txt"]:
  files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>